In [1]:
import pandas as pd
df1 = pd.read_csv(r'C:\Users\brian\Desktop\JUNSOO\exhibition\data\통합2.csv' , encoding = 'cp949')
df2 = pd.read_csv(r'C:\Users\brian\Desktop\JUNSOO\exhibition\data\final_data.csv' , encoding = 'cp949')

# 전체학생수가 0인학교 지우기
df1 = df1.loc[df1['전체학생수']!= 0 , :]

# 학교점수에 반영되지 않은 컬럼중 교육비지표와 연관된다 판단한 변수들
col_pay = [
    'key' , '전체학생수' , '상담실적(외부상담전문가)'
    , '연간1인당 보건실이용건수' , '교수학습공간 일반교실'
    , '교수학습공간 특별교실' 
    , '교수학습공간 수준별교실' ,'참여학생수-동아리?학생자치활동' , '참여학생수-또래활동' 
           , '참여학생수-교육주간 활동' , '참여학생수-기타 학교폭력 예방활동'
    
]       

df1 = df1[col_pay]

In [2]:
df2.head().T
df2.replace(["유","무","시설없음"], [1,0,0], inplace = True) 
df2.replace(["적정설치","단순설치","미설치"], [2,1,0], inplace = True) 
df2.head().T

C:\Users\brian\AppData\Local\Temp\ipykernel_10060\1006941924.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.replace(["유","무","시설없음"], [1,0,0], inplace = True)
C:\Users\brian\AppData\Local\Temp\ipykernel_10060\1006941924.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.replace(["적정설치","단순설치","미설치"], [2,1,0], inplace = True)


,0,1,2,3,4
key,S000003511-2021,S000003511-2022,S000003511-2023,S000003511-2024,S000003514-2021
지역,서울특별시 서초구,서울특별시 서초구,서울특별시 서초구,서울특별시 서초구,서울특별시 성북구
정보공시 학교코드,S000003511,S000003511,S000003511,S000003511,S000003514
학교명,서울교육대학교부설초등학교,서울교육대학교부설초등학교,서울교육대학교부설초등학교,서울교육대학교부설초등학교,서울대학교사범대학부설중학교
x1,0,0,0,0,0
x2,0,0,0,0,0
x3,0,0,0,0,0
x4,19.8,19.8,19.2,19.2,11.3
x5,2.705863,4.518985,5.001961,4.854098,2.66
x6,0,0,0,0,1


In [3]:
#교육비지표 산출
df1['x1'] = df1['전체학생수']
df1['x2'] = (df1['참여학생수-동아리?학생자치활동'] + 
             df1['참여학생수-또래활동'] + df1['참여학생수-교육주간 활동'] + df1['참여학생수-기타 학교폭력 예방활동'])
df1['x3'] = (df1['교수학습공간 특별교실'] + df1['교수학습공간 수준별교실'])
df1['x4'] = df1['연간1인당 보건실이용건수']
df1['x5'] = df1['상담실적(외부상담전문가)']

# 학교점수 산정 변수 포함

df2['c1'] = df2['x5'] #동아리 다양성
df2['c2'] = df2['x7'] #자율학교 운영여부
#df1.drop([] ,axis = 1, inplace = True)   # drop 에서 axis?

In [4]:
school_score = df2['y']
school_score = pd.DataFrame(school_score)
school_score.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5456 entries, 0 to 5455
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   y       5456 non-null   float64
dtypes: float64(1)
memory usage: 42.8 KB


In [5]:
# 변수들 + key만 선택한 데이터셋
pay_data = df1.loc[:,"x1":"x5"].copy()
schooltopay_data = df2.loc[:,'c1':'c2']


In [6]:
# 합치기
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

pay_data["y2"] = scaler.fit_transform(pay_data).sum(axis = 1).tolist()
schooltopay_data["y3"] = scaler.fit_transform(schooltopay_data).sum(axis = 1).tolist()

In [11]:
pay_score = pay_data["y2"] + schooltopay_data["y3"]
pay_score.info()

<class 'pandas.core.series.Series'>
Index: 5470 entries, 0 to 5474
Series name: None
Non-Null Count  Dtype  
--------------  -----  
5442 non-null   float64
dtypes: float64(1)
memory usage: 85.5 KB


In [12]:
keydata = pd.DataFrame(df1['key'])
payAk = pd.concat([schooltopay_data , keydata] , axis = 1)
payAk.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5470 entries, 0 to 5474
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   c1      5456 non-null   float64
 1   c2      5456 non-null   float64
 2   y3      5456 non-null   float64
 3   key     5456 non-null   object 
dtypes: float64(3), object(1)
memory usage: 213.7+ KB


In [13]:
plzcorr2 = pd.concat([payAk , school_score] , axis = 1)

In [14]:
plzcorr2.drop(columns= 'key' , inplace = True)
plzcorr2.info()
plzcorr2.corr(method= 'pearson')

<class 'pandas.core.frame.DataFrame'>
Index: 5470 entries, 0 to 5474
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   c1      5456 non-null   float64
 1   c2      5456 non-null   float64
 2   y3      5456 non-null   float64
 3   y       5456 non-null   float64
dtypes: float64(4)
memory usage: 213.7 KB


,c1,c2,y3,y
c1,1.000000,-0.038947,0.039179,0.157060
c2,-0.038947,1.000000,0.996948,0.213433
y3,0.039179,0.996948,1.000000,0.225701
y,0.157060,0.213433,0.225701,1.000000
